In [21]:
from pathlib import Path
import pandas as pd

BASE_OUTPUTS_DIR = Path("outputs/")

LANGUAGES = ["acehnese", "javanese", "indonesian", "hitu", "minangkabau", "vietnamese"]
PROMPT_TYPES = ["few_shots_only", "few_shots_and_word_candidates", "topic_related"]

def find_per_model_output_files(lang: str, prompt_type: str) -> list:
    outputs_dir = BASE_OUTPUTS_DIR / prompt_type
    pattern = f"{lang}_{prompt_type}_*.csv"
    files = sorted(outputs_dir.glob(pattern))
    files = [f for f in files if not f.name.endswith(".bak")]
    return files


def load_all_rows() -> pd.DataFrame:
    all_dfs = []

    for prompt_type in PROMPT_TYPES:
        for lang in LANGUAGES:
            files = find_per_model_output_files(lang, prompt_type)

            if not files:
                print(f"[{lang}/{prompt_type}] no files found, skipping")
                continue

            for f in files:
                df = pd.read_csv(f)
                df["language"] = lang
                df["prompt_type"] = prompt_type

                if "model" not in df.columns or "is_thinking_on" not in df.columns:
                    stem = f.stem
                    prefix = f"{lang}_{prompt_type}_"
                    suffix = stem[len(prefix):] if stem.startswith(prefix) else stem
                    if "model" not in df.columns:
                        df["model"] = suffix.rsplit("_thinking_", 1)[0]
                    if "is_thinking_on" not in df.columns:
                        df["is_thinking_on"] = suffix.endswith("_on")

                all_dfs.append(df)

    if not all_dfs:
        raise RuntimeError("No output files found -- check BASE_OUTPUTS_DIR / LANGUAGES / PROMPT_TYPES.")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined


combined = load_all_rows()
print(f"Loaded {len(combined)} total rows")

combined.head()

Loaded 54000 total rows


,prompt_id,language,length_category,unit,target_length_desc,prompt_type,topic,prompt,output_id,output,...,full_response,model,is_thinking_on,is_valid,fluency_score,cultural_relevance_score,pct_loan_words,seed,candidate_seed,sentence_match_rate
0,b8641c12-b541-45b1-bde2-740053f3baaa,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,8d21c92e-0625-4b69-9aec-825f4a44c2c5,Geudubng meu kawan that.,...,"{""id"": ""msg_011CecP26ARnddtBSBQ7ar7p"", ""contai...",claude-haiku-4-5,False,True,2.0,0.0,0.25,NaN,NaN,NaN
1,a1b43170-baf5-430e-9d58-287c1b3be16e,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,77ebae05-7d89-4416-a10b-7c27b90a2347,Gaki nyoe makan nasi goreng.,...,"{""id"": ""msg_011CecP26BfYdtaQj13QDope"", ""contai...",claude-haiku-4-5,False,True,1.0,4.0,0.20,NaN,NaN,NaN
2,ae51a88a-5190-4e4e-a30b-42c96c3593c3,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,96f96e61-fbd7-4ce8-8421-ae5444f0a964,Gajah nyoe cukôp besar banget.,...,"{""id"": ""msg_011CecP268gvBTPSatj1xHez"", ""contai...",claude-haiku-4-5,False,True,3.0,0.0,0.40,NaN,NaN,NaN
3,4e57ed68-22e6-426d-b1cc-c2cb38751fc0,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,25965002-c1e9-4873-8a50-b19919acd8cf,Geucok meu babah that nyoe.,...,"{""id"": ""msg_011CecP26GNy1W1sr6gbsWgF"", ""contai...",claude-haiku-4-5,False,True,1.0,0.0,0.00,NaN,NaN,NaN
4,3facf64f-3224-4940-a17a-37c9a41559a4,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,a899a39e-8eba-4bc0-a78e-f14d61c3017d,Rumoh meuë gampong that likôt.,...,"{""id"": ""msg_011CecP26CucoV69AQRWhNf2"", ""contai...",claude-haiku-4-5,False,True,1.0,0.0,0.20,NaN,NaN,NaN


In [22]:
import re
from pathlib import Path
import pandas as pd

# ==============================================================
# CONFIGURATION
# ==============================================================
DICTIONARIES_DIR = Path("../dictionaries")
OUTPUT_SENTENCE_COL = "output"

TOKEN_PATTERN = re.compile(r"[^\W\d_]+(?:'[^\W\d_]+)*", flags=re.UNICODE)


# ==============================================================
# 1. HELPER FUNCTIONS
# ==============================================================
def tokenize(text: str):
    if not isinstance(text, str):
        return []
    return TOKEN_PATTERN.findall(text.lower())


def build_dictionary_word_set(dictionary_csv, lemma_cols=("word",)) -> set:
    """
    Build a flat, tokenized set of all valid dictionary word forms.
    """
    dict_words = set()
    if not dictionary_csv.exists():
        print(f"Warning: Dictionary not found at {dictionary_csv}")
        return dict_words
        
    dict_df = pd.read_csv(dictionary_csv)
    for col in lemma_cols:
        if col not in dict_df.columns:
            continue
        for val in dict_df[col].dropna():
            dict_words |= set(tokenize(val))

    return dict_words


def calc_pct_loan_words(text, dictionary_words):
    """
    Calculates the percentage of tokens in the text that are NOT in the dictionary.
    """
    tokens = tokenize(text)
    if not tokens:
        return 0.0

    not_in_dict = sum(1 for t in tokens if t not in dictionary_words)
    return round(not_in_dict / len(tokens), 4)


# ==============================================================
# 2. APPLY CALCULATION TO DATAFRAME
# ==============================================================
print("Loading dictionaries and calculating pct_loan_words...")

dict_cache = {}

def get_pct_loan(row):
    lang = row["language"]
    
    # Load and cache dictionary on first access to save IO time
    if lang not in dict_cache:
        dict_path = DICTIONARIES_DIR / f"{lang}.csv"
        dict_cache[lang] = build_dictionary_word_set(dict_path)
        
    return calc_pct_loan_words(row[OUTPUT_SENTENCE_COL], dict_cache[lang])

# Calculate metric for every row
combined["pct_loan_words"] = combined.apply(get_pct_loan, axis=1)

Loading dictionaries and calculating pct_loan_words...


In [23]:
# ==============================================================
# 3. AGGREGATE RESULTS
# ==============================================================
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# combined.to_csv(RESULTS_DIR / "all_off_target_rate.csv", index=False)

# --- Group 1: Language and Prompt Type ---
group_cols_1 = ["language", "prompt_type"]

loan_summary_1 = (
    combined.groupby(group_cols_1, dropna=False)
    .agg(
        n_outputs=("pct_loan_words", "size"),
        off_target_rate=("pct_loan_words", "mean")
    )
    .reset_index()
)

print("\n=== SUMMARY 1: LANGUAGE & PROMPT TYPE ===")
display(loan_summary_1)
loan_summary_1.to_csv(RESULTS_DIR / "overall_off_target_rate.csv", index=False)

# --- Group 2: Language, Model, Thinking, and Prompt Type ---
group_cols_2 = ["language", "model", "is_thinking_on", "prompt_type"]

loan_summary_2 = (
    combined.groupby(group_cols_2, dropna=False)
    .agg(
        n_outputs=("pct_loan_words", "size"),
        off_target_rate=("pct_loan_words", "mean")
    )
    .reset_index()
)

print("\n=== SUMMARY 2: LANGUAGE, MODEL, THINKING & PROMPT TYPE ===")
display(loan_summary_2)
loan_summary_2.to_csv(RESULTS_DIR / "detailed_off_target_rate.csv", index=False)

print(f"\nSaved detailed and summary CSVs to {RESULTS_DIR}/")


=== SUMMARY 1: LANGUAGE & PROMPT TYPE ===


,language,prompt_type,n_outputs,off_target_rate
0,acehnese,few_shots_and_word_candidates,3000,0.064067
1,acehnese,few_shots_only,3000,0.315004
2,acehnese,topic_related,3000,0.413191
3,hitu,few_shots_and_word_candidates,3000,0.059706
4,hitu,few_shots_only,3000,0.608707
5,hitu,topic_related,3000,0.835725
6,indonesian,few_shots_and_word_candidates,3000,0.014059
7,indonesian,few_shots_only,3000,0.018400
8,indonesian,topic_related,3000,0.028208
9,javanese,few_shots_and_word_candidates,3000,0.090210



=== SUMMARY 2: LANGUAGE, MODEL, THINKING & PROMPT TYPE ===


,language,model,is_thinking_on,prompt_type,n_outputs,off_target_rate
0,acehnese,claude-haiku-4-5,False,few_shots_and_word_candidates,500,0.035541
1,acehnese,claude-haiku-4-5,False,few_shots_only,500,0.403164
2,acehnese,claude-haiku-4-5,False,topic_related,500,0.545599
3,acehnese,claude-haiku-4-5,True,few_shots_and_word_candidates,500,0.052139
4,acehnese,claude-haiku-4-5,True,few_shots_only,500,0.395138
...,...,...,...,...,...,...
103,vietnamese,gpt-5.4-mini,False,few_shots_only,500,0.001158
104,vietnamese,gpt-5.4-mini,False,topic_related,500,0.014151
105,vietnamese,gpt-5.4-mini,True,few_shots_and_word_candidates,500,0.000913
106,vietnamese,gpt-5.4-mini,True,few_shots_only,500,0.000825



Saved detailed and summary CSVs to results/
